# S0_plan (local CPU): hashes, the three data sets, disjointness, plan.json

In [ ]:
import glob, os, json, hashlib, sqlite3, time
from pathlib import Path
from collections import Counter
MYDRIVE = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive"
EZ = MYDRIVE / "ez-penta"
PT3 = EZ / "post_train_3"
OUTD = PT3 / "steering"
DB = EZ / "dataset2_eval_gguf" / "combined" / "results.sqlite3"


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]


def folder_ok(d):
    if (d / "DONE").read_text().strip() != sha256_file(d / "manifest.json"):
        return False
    man = json.loads((d / "manifest.json").read_text())
    return all(sha256_file(d / f) == m["sha256"] for f, m in man["files"].items())


# 1. hashes
HASHES = {}
HASHES["results.sqlite3"] = sha256_file(DB)
assert HASHES["results.sqlite3"].startswith("ecffa936c76c1009"), HASHES["results.sqlite3"]
HASHES["frozen_pool.json"] = sha256_file(PT3 / "arm_en" / "frozen_pool.json")
jplan = json.loads((PT3 / "japan_culture" / "plan.json").read_text())
assert (PT3 / "japan_culture" / "plan.DONE").read_text().strip() == sha256_file(PT3 / "japan_culture" / "plan.json")
assert HASHES["frozen_pool.json"] == jplan["frozen_pool_sha256"], "frozen_pool.json differs from the hash recorded by the Japan-culture plan"
frozen = json.loads((PT3 / "arm_en" / "frozen_pool.json").read_text())
BATCHES = sorted(frozen["batches"])
assert BATCHES == [1, 3, 5, 7, 9, 11], BATCHES
for b in BATCHES:
    bd = PT3 / "eval_pool" / "batches" / f"b{b:04d}"
    assert folder_ok(bd), f"batch {b} hash check failed"
    HASHES[f"batch_b{b:04d}_manifest"] = sha256_file(bd / "manifest.json")
    assert HASHES[f"batch_b{b:04d}_manifest"] == jplan["batch_manifest_sha256"][str(b)]
    ev = PT3 / "arm_en" / "eval" / f"b{b:04d}.jsonl"
    eman = json.loads((PT3 / "arm_en" / "eval" / f"b{b:04d}.manifest.json").read_text())
    HASHES[f"arm_en_eval_b{b:04d}"] = sha256_file(ev)
    assert HASHES[f"arm_en_eval_b{b:04d}"] == eman["sha256"] == jplan["us_answer_sources"][str(b)]["eval_sha256"]
    for sub in ("prompts", "base", "en_s0"):
        cd = PT3 / "japan_culture" / "chunks" / f"c{b:04d}" / sub
        assert folder_ok(cd), f"japan chunk {b}/{sub} hash check failed"
        HASHES[f"jp_c{b:04d}_{sub}_manifest"] = sha256_file(cd / "manifest.json")
print(json.dumps(HASHES, indent=1))


In [ ]:
# 2. Extraction (analysis) and selection (heldout) sets from Dataset2.
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
q = """select prompt_id, seed_id, split, category, severity, kind, culture, framing, prompt_language, prompt, identical_twin_flag
       from observations where split = ? and framing in ('control','fc','rp') and prompt_language in ('en','ja')
       and identical_twin_flag = 0 order by prompt_id"""
def dset(split):
    rows = con.execute(q, (split,)).fetchall()
    cols = ["prompt_id", "seed_id", "split", "category", "severity", "kind", "culture", "framing", "lang", "prompt", "identical_twin_flag"]
    out = [dict(zip(cols, r)) for r in rows]
    for r in out:
        del r["identical_twin_flag"]
    return out
EXTRACTION = dset("analysis")
SELECTION = dset("heldout")
n_flag = {s: con.execute("select count(*) from observations where split=? and framing in ('control','fc','rp') and prompt_language in ('en','ja') and identical_twin_flag=1", (s,)).fetchone()[0] for s in ("analysis", "heldout")}
con.close()
print("extraction", len(EXTRACTION), "selection", len(SELECTION), "dropped identical twins:", n_flag)
for name, S in (("extraction", EXTRACTION), ("selection", SELECTION)):
    print(name, Counter((r["kind"], r["severity"]) for r in S), Counter(r["culture"] for r in S), Counter(r["lang"] for r in S))


In [ ]:
# 3. Final-evaluation set: US = frozen pool, first occurrence of each seed walking the batches in ascending order;
#    Japan = Japanese-culture prompts of paired_ok seeds (all 6 chunks).
FSEEDS = set(frozen["seed_ids"])
assert len(FSEEDS) == 519
EVAL_US, seen, us_batch = [], set(), {}
for b in BATCHES:
    bd = PT3 / "eval_pool" / "batches" / f"b{b:04d}"
    ps = [p for p in load_jsonl(bd / "prompts.jsonl") if p["seed_id"] in FSEEDS and p["seed_id"] not in seen]
    new = {p["seed_id"] for p in ps}
    for p in ps:
        EVAL_US.append({k: p[k] for k in ("prompt_id", "seed_id", "batch", "category", "subtopic", "form", "severity", "kind", "expected",
                                          "culture", "framing", "lang", "prompt")})
        us_batch[p["prompt_id"]] = b
    seen |= new
assert len(seen) == 519 and len(EVAL_US) == 6228 and len({p["prompt_id"] for p in EVAL_US}) == 6228, (len(seen), len(EVAL_US))
assert all(p["culture"] == "us" for p in EVAL_US)
first_batch = {p["seed_id"]: p["batch"] for p in EVAL_US}
EVAL_JP, OK = [], set()
for b in BATCHES:
    cd = PT3 / "japan_culture" / "chunks" / f"c{b:04d}" / "prompts"
    ok = {c["seed_id"] for c in load_jsonl(cd / "seed_checks.jsonl") if c["paired_ok"]}
    OK |= ok
    for p in load_jsonl(cd / "prompts_japan.jsonl"):
        if p["seed_id"] in ok:
            assert first_batch[p["seed_id"]] == b, "Japan chunk differs from the seed's first frozen batch"
            EVAL_JP.append({**{k: p[k] for k in ("prompt_id", "seed_id", "batch", "category", "subtopic", "form", "severity", "kind", "expected",
                                                 "culture", "framing", "lang", "prompt", "us_prompt_id")}, "chunk": b})
assert len(EVAL_JP) == 12 * len(OK), (len(EVAL_JP), len(OK))
assert len({p["prompt_id"] for p in EVAL_JP}) == len(EVAL_JP) and not ({p["prompt_id"] for p in EVAL_JP} & set(us_batch))
print("eval US", len(EVAL_US), "eval JP", len(EVAL_JP), "paired_ok seeds", len(OK))

# Disjointness (0.2).
ext_seeds = {r["seed_id"] for r in EXTRACTION}
sel_seeds = {r["seed_id"] for r in SELECTION}
eval_seeds = {r["seed_id"] for r in EVAL_US} | {r["seed_id"] for r in EVAL_JP}
assert not (eval_seeds & ext_seeds), "evaluation seed in extraction"
assert not (eval_seeds & sel_seeds), "evaluation seed in selection"
assert not (ext_seeds & sel_seeds), "extraction and selection share a seed"
ext_ids = {r["prompt_id"] for r in EXTRACTION}
sel_ids = {r["prompt_id"] for r in SELECTION}
eval_ids = {r["prompt_id"] for r in EVAL_US} | {r["prompt_id"] for r in EVAL_JP}
assert not (eval_ids & (ext_ids | sel_ids)) and not (ext_ids & sel_ids)
print("disjointness asserts passed: extraction", len(ext_seeds), "seeds; selection", len(sel_seeds), "seeds; evaluation", len(eval_seeds), "seeds")


In [ ]:
# 4. Fixed halves and plan.json
def half(seed_id):
    return int(hashlib.sha256(seed_id.encode()).hexdigest(), 16) % 2 == 0
sel_half = sorted(s for s in sel_seeds if half(s))
eval_half = sorted(s for s in eval_seeds if half(s))
print("selection half seeds", len(sel_half), "of", len(sel_seeds), "| prompts", sum(r["seed_id"] in set(sel_half) for r in SELECTION))
print("evaluation half seeds", len(eval_half), "of", len(eval_seeds))
plan = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "hashes": HASHES,
        "filters": {"extraction": "split=analysis, framing in (control, fc, rp), language in (en, ja), identical_twin_flag=0",
                    "selection": "split=heldout, same filter", "dropped_identical_twins": n_flag},
        "counts": {"extraction": len(EXTRACTION), "selection": len(SELECTION), "eval_us": len(EVAL_US), "eval_jp": len(EVAL_JP),
                   "paired_ok_seeds": len(OK)},
        "sets": {"extraction": EXTRACTION, "selection": SELECTION, "eval_us": EVAL_US, "eval_jp": EVAL_JP},
        "us_first_batch": us_batch, "selection_half_seeds": sel_half, "eval_half_seeds": eval_half,
        "heldout_used_for_selection": True}
OUTD.mkdir(parents=True, exist_ok=True)
(OUTD / "plan.json").write_text(json.dumps(plan, ensure_ascii=False))
(OUTD / "plan.DONE").write_text(sha256_file(OUTD / "plan.json") + "\n")
print("plan.json", (OUTD / "plan.json").stat().st_size, "bytes; sha256", sha256_file(OUTD / "plan.json"))
